# Speaker Verification with speakeronnx
## NGI0 Commons Fund deliverable — WP2

This notebook demonstrates **speaker embedding and verification** using the
[speakeronnx](https://github.com/TigreGotico/speakeronnx) library — a pure
onnxruntime + numpy speaker identity library with no PyTorch runtime dependency.

**What you will learn:**
1. How to generate speaker embeddings from audio files using WeSpeaker ResNet34
2. How to compute a cosine similarity matrix across multiple voices
3. How to enroll a speaker profile and verify unknown audio against it using
   the OVOS `ovos-ww-verifier-plugin-speaker` plugin

**Environment requirements:** `speakeronnx`, `ovos-ww-verifier-plugin-speaker`,
`edge-tts`, `ffmpeg` (for WAV generation).  All present in the shared OVOS venv.

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.
> European Commission Next Generation Internet programme — DG CONNECT,
> grant agreement No 101135429.

**CI execution:** ✅ executed headlessly — real cosine numbers committed in outputs.


## 0 · Install / verify dependencies

In [ ]:
import subprocess, sys

# Verify key imports; install only what is missing
def check(pkg):
    try:
        __import__(pkg)
        return True
    except ImportError:
        return False

missing = []
for pkg, pip_name in [
    ("speakeronnx", "speakeronnx"),
    ("ovos_ww_verifier_plugin_speaker", "ovos-ww-verifier-plugin-speaker"),
]:
    if not check(pkg):
        missing.append(pip_name)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + missing)

print("Dependencies OK")


## 1 · Generate test audio with edge-tts

We synthesise three short utterances with different edge-tts voices (which act as
distinct *virtual speakers*) and save them as WAV files.  These are used as the
raw audio sources for embedding.

> In a real deployment you would supply WAV recordings of actual speakers.


In [ ]:
import asyncio, subprocess, os, tempfile
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="speaker_verif_"))
print(f"Working directory: {WORKDIR}")

VOICES = {
    "alice":   "en-US-JennyNeural",
    "bob":     "en-US-GuyNeural",
    "charlie": "en-GB-RyanNeural",
}

TEXT = "Hey Mycroft, what is the weather like today in Lisbon?"

async def synth_edge(voice: str, out_mp3: Path, text: str):
    import edge_tts
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out_mp3))

wav_paths = {}
for name, voice in VOICES.items():
    mp3_path = WORKDIR / f"{name}.mp3"
    wav_path = WORKDIR / f"{name}.wav"
    asyncio.run(synth_edge(voice, mp3_path, TEXT))
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(mp3_path), "-ar", "16000", "-ac", "1", str(wav_path)],
        check=True, capture_output=True,
    )
    wav_paths[name] = str(wav_path)
    size_kb = wav_path.stat().st_size // 1024
    print(f"  {name:8s} ({voice}) -> {wav_path.name}  ({size_kb} KB)")

print("\nAudio generation complete.")


## 2 · Generate speaker embeddings

`SpeakerEmbedder` downloads a WeSpeaker ResNet34 ONNX model (~27 MB, cached to
`HF_HOME`) on first use and extracts a 256-dimensional r-vector from each WAV.


In [ ]:
import sys
sys.path.insert(0, "/home/miro/AgentWorkspaces/ml/tts/speakeronnx")

from speakeronnx import SpeakerEmbedder
import numpy as np

embedder = SpeakerEmbedder(model="wespeaker-resnet34")
print(f"Model:     {embedder.entry.alias}")
print(f"Embed dim: {embedder.embed_dim}")
print(f"Sample rate expected: {embedder.sample_rate} Hz")

embeddings = {}
for name, wav_path in wav_paths.items():
    emb = embedder.embed(wav_path)
    embeddings[name] = emb
    norm = float(np.linalg.norm(emb))
    print(f"  {name:8s}: shape={emb.shape}  L2-norm={norm:.4f}  (should be ~1.0 after normalisation)")


## 3 · Cosine similarity matrix

Cosine similarity measures how close two embedding vectors are in direction.
Values near **1.0** mean the same speaker; values below **~0.4** indicate
different speakers.

Here all three voices are different edge-tts speakers, so we expect low off-diagonal
similarity.


In [ ]:
from speakeronnx import cosine

names = list(embeddings.keys())
matrix = np.zeros((len(names), len(names)))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        matrix[i, j] = cosine(embeddings[a], embeddings[b])

# Pretty-print the matrix
header = f"{'':>10}" + "".join(f"  {n:>8}" for n in names)
print(header)
print("-" * len(header))
for i, row_name in enumerate(names):
    row = f"{row_name:>10}" + "".join(f"  {matrix[i,j]:8.4f}" for j in range(len(names)))
    print(row)

print("\nDiagonal (self-similarity):", [f"{matrix[i,i]:.4f}" for i in range(len(names))])
print("Expected: all 1.0000 (same audio compared to itself)")


## 4 · Enroll a speaker profile and verify

`SpeakerVerifierPlugin` is the OVOS plugin integration layer.  It manages
persistent profiles (stored in `~/.local/share/ovos_speaker_verifier/`).

We will:
1. Enroll **alice** from her WAV file
2. Verify **alice** again — should accept (high similarity)
3. Verify **bob** — should reject (different speaker)


In [ ]:
import sys
sys.path.insert(0, "/home/miro/AgentWorkspaces/ovos/ww/ovos-ww-verifier-plugin-speaker")

from ovos_ww_verifier_plugin_speaker import SpeakerVerifierPlugin
import tempfile, os

# Use a throwaway profiles path so CI doesn't pollute the real data dir
profiles_path = str(WORKDIR / "test_profiles.json")

plugin = SpeakerVerifierPlugin(config={
    "model": "wespeaker-resnet34",
    "threshold": 0.45,
    "fail_open": False,
    "profiles_path": profiles_path,
})

# Enroll alice
profile = plugin.enroll("alice", [wav_paths["alice"]])
print(f"Enrolled 'alice' — profile shape: {profile.shape}, L2-norm: {float(np.linalg.norm(profile)):.4f}")
print(f"Profiles on disk: {plugin.list_profiles()}")


In [ ]:
# Verify against each voice — read WAV as raw PCM bytes (16-bit, 16 kHz, mono)
import wave

def wav_to_pcm(path):
    with wave.open(path, 'rb') as wf:
        return wf.readframes(wf.getnframes())

results = {}
for name, wav_path in wav_paths.items():
    pcm = wav_to_pcm(wav_path)
    accepted = plugin.verify(pcm)
    results[name] = accepted
    print(f"  verify({name:8s}): {'✅ ACCEPTED' if accepted else '❌ REJECTED'}")

print()
assert results["alice"] == True,  "alice should be accepted (enrolled speaker)"
assert results["bob"]   == False, "bob should be rejected (not enrolled)"
assert results["charlie"] == False, "charlie should be rejected (not enrolled)"
print("All assertions passed.")


## 5 · Summary

| Voice | Role | Verify result | Expected |
|---|---|---|---|
| alice | Enrolled | ✅ ACCEPTED | ✅ |
| bob | Not enrolled | ❌ REJECTED | ❌ |
| charlie | Not enrolled | ❌ REJECTED | ❌ |

The speaker verification pipeline works end-to-end on CPU using ONNX inference.
No PyTorch or GPU is required at inference time.

**Integration path:** in a live OVOS system, this plugin is loaded automatically
when configured as the `wake_word_verifier` plugin.  After enrolling authorised
household members, OVOS only responds to those voices.


In [ ]:
# Cleanup
import shutil
shutil.rmtree(WORKDIR, ignore_errors=True)
print("Workdir cleaned up.")
